# 🍃 Bengaluru Air Quality Index (AQI) — Data Audit & ML Modeling
This notebook walks through the data audit, environmental feature engineering, baseline persistence benchmarking, and machine learning model training for the Bengaluru AQI platform.

Reusable data processing, models, evaluation, and inference engines have been modularized under `src/`.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import the modular src package
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if "notebooks" in str(NOTEBOOK_DIR) else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.cpcb_loader import aggregate_raw_directory, audit_station_records
from src.preprocessing.cleaner import sanitize_sensor_readings
from src.preprocessing.features import (
    MODEL_FEATURES,
    TARGET_DIRECT,
    build_time_features,
    generate_lag_features,
    generate_target_column,
)
from src.models.trainer import train_model, save_pipeline
from src.evaluation.metrics import (
    calculate_regression_metrics,
    evaluate_predictions_by_level,
    evaluate_persistence_baseline,
)

print("Modules from src loaded successfully!")
print("Project root:", PROJECT_ROOT)

## 1. Data Ingestion & Audit
We aggregate the raw CPCB station monitoring CSV files (BTM Layout, Jayanagar, Silk Board, Peenya) and audit missing values and continuity.

In [ ]:
raw_data_dir = PROJECT_ROOT / "data" / "raw_cpcb"
master_df = aggregate_raw_directory(raw_data_dir)
print(f"Total raw records loaded: {len(master_df):,}")
display(master_df.head())

In [ ]:
audit_info = audit_station_records(master_df)
print("Monitored Stations:", audit_info["stations"])
print(f"Date range: {audit_info['min_time']} to {audit_info['max_time']}")

# Station-wise record counts
print("
Records per station:")
print(master_df["Station"].value_counts())

## 2. Sensor Telemetry Sanitation & Outlier Handling
Hardware fault diagnostic values (e.g. 999.99, -999) and out-of-range environmental readings are filtered using domain-specific bounds.

In [ ]:
clean_df = sanitize_sensor_readings(master_df)
print("Sanitized dataset shape:", clean_df.shape)

## 3. Feature Engineering: Temporal Cycles & Multi-Hour Lags
We compute time features (Hour, DayOfWeek, Month, Day) and generate 1h, 3h, 6h, and 24h lag features for PM2.5 partitioned per station.

In [ ]:
clean_df = build_time_features(clean_df, timestamp_col="Timestamp")
clean_df = clean_df.sort_values(["Station", "Timestamp"]).reset_index(drop=True)
clean_df = generate_lag_features(clean_df, station_col="Station", lags=[1, 3, 6, 24])
clean_df = generate_target_column(clean_df, station_col="Station", lead=1)

# Filter complete cases for the modeling feature set
ml_df = clean_df.dropna(subset=[TARGET_DIRECT, "PM2.5 (µg/m³)", "PM25_lag_1h"]).copy()
print(f"Final ML-ready observations: {len(ml_df):,}")

## 4. Train / Test Temporal Separation
To simulate realistic forecasting without data leakage:
- **Training Set**: 2019 – 2023 observations
- **Test Set**: 2024 observations

In [ ]:
train_df = ml_df[ml_df["Year"].between(2019, 2023)].copy()
test_df = ml_df[ml_df["Year"] == 2024].copy()

print(f"Training observations: {len(train_df):,}")
print(f"Test observations:     {len(test_df):,}")

X_train = train_df[MODEL_FEATURES]
y_train_direct = train_df[TARGET_DIRECT]
y_train_change = train_df[TARGET_DIRECT] - train_df["PM2.5 (µg/m³)"]

X_test = test_df[MODEL_FEATURES]
y_test_direct = test_df[TARGET_DIRECT]
y_test_change = test_df[TARGET_DIRECT] - test_df["PM2.5 (µg/m³)"]

## 5. Model Benchmarking: Baseline vs Direct Regressors
We benchmark the naive persistence baseline against Random Forest Direct and XGBoost Direct models.

In [ ]:
# Naive Persistence Baseline
persistence_metrics = evaluate_persistence_baseline(
    y_current=test_df["PM2.5 (µg/m³)"].values,
    y_next=y_test_direct.values,
)
print("Persistence Baseline:", persistence_metrics)

In [ ]:
# Train Random Forest Direct
print("Training Random Forest Direct...")
rf_direct = train_model(
    model_type="random_forest",
    approach="direct",
    X_train=X_train,
    y_train=y_train_direct,
    n_estimators=100,
    max_depth=12,
)
pred_rf = rf_direct.predict(X_test)
rf_metrics = calculate_regression_metrics(y_test_direct.values, pred_rf)
print("Random Forest Direct:", rf_metrics)

In [ ]:
# Train XGBoost Direct
print("Training XGBoost Direct...")
xgb_direct = train_model(
    model_type="xgboost",
    approach="direct",
    X_train=X_train,
    y_train=y_train_direct,
    n_estimators=150,
    max_depth=6,
)
pred_xgb = xgb_direct.predict(X_test)
xgb_metrics = calculate_regression_metrics(y_test_direct.values, pred_xgb)
print("XGBoost Direct:", xgb_metrics)

## 6. Change / Delta Prediction Architecture
Instead of predicting absolute PM2.5 directly, the Change architecture predicts the delta $\Delta = \text{PM2.5}_{t+1} - \text{PM2.5}_t$.
HistGradientBoosting and Random Forest Change models significantly improve performance on rapid pollution transitions.

In [ ]:
print("Training HistGradientBoosting Change...")
hgb_change = train_model(
    model_type="hist_gradient_boosting",
    approach="change",
    X_train=X_train,
    y_train=y_train_change,
    max_iter=150,
    max_depth=8,
)
pred_delta_hgb = hgb_change.predict(X_test)
pred_pm25_hgb = pred_delta_hgb + test_df["PM2.5 (µg/m³)"].values
hgb_metrics = calculate_regression_metrics(y_test_direct.values, pred_pm25_hgb)
print("HistGradientBoosting Change:", hgb_metrics)

## 7. Model Comparison Summary & Visualizations

In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "Persistence Baseline", **persistence_metrics},
    {"Model": "Random Forest Direct", **rf_metrics},
    {"Model": "XGBoost Direct", **xgb_metrics},
    {"Model": "HistGradientBoosting Change", **hgb_metrics},
])
display(comparison_df)

In [ ]:
# Actual vs Predicted PM2.5 Scatter Plot
plt.figure(figsize=(10, 5))
plt.scatter(y_test_direct.values, pred_pm25_hgb, alpha=0.3, s=8, color="#0ea5e9", label="Predictions")
max_val = max(y_test_direct.max(), pred_pm25_hgb.max())
plt.plot([0, max_val], [0, max_val], 'r--', lw=1.5, label="Perfect Prediction")
plt.xlabel("Actual PM2.5 (µg/m³)")
plt.ylabel("Predicted PM2.5 (µg/m³)")
plt.title("HistGradientBoosting (Change Model): Actual vs Predicted PM2.5")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Tiered Performance by PM2.5 Severity
Performance breakdown across CPCB NAQI severity bands.

In [ ]:
tiered_results = evaluate_predictions_by_level(y_test_direct.values, pred_pm25_hgb)
tiered_df = pd.DataFrame(tiered_results)
display(tiered_df)